# Practical Lab — Local AI with Ollama
**Running and Developing Local AI Applications** · PHD Student 

**Prerequisites:** Python 3.10+, basic Python, basic understanding of APIs, 8–16 GB RAM (NVIDIA GPU optional).

**Technologies:** Ollama · Python · REST API · `requests` · (optional) Open WebUI · a local LLM (Qwen, Gemma, Llama…)

> Ollama exposes a local HTTP service on `localhost:11434`, so the same model can be used from the terminal, Python, or any other software.

### How to use this notebook
- Run cells top to bottom. Cells starting with `!` run shell commands.
- Change `MODEL` / `MODEL2` below if the model names differ on your installation.
- Fill in the **✍️ Your answer** cells — they feed your final report.

## Part 1 — Understanding Local AI

**Cloud AI**
```
Python App → Internet → Cloud API → LLM
```
**Local AI**
```
Python App → localhost:11434 → Ollama → Local LLM → GPU / CPU
```
Advantages to investigate: **privacy**, **offline operation**, **no per-token cost**.
Trade-off: local models may be **less capable or slower** than frontier cloud models, depending on hardware and model size.

## Part 2 — Installing Ollama
### Exercise 1 — Verify the environment

In [2]:
import sys, platform
print("Python  :", sys.version)
print("System  :", platform.system(), platform.release())
assert sys.version_info >= (3, 10), "Python 3.10+ required"

Python  : 3.13.5 (tags/v3.13.5:6cb20a2, Jun 11 2025, 16:15:46) [MSC v.1943 64 bit (AMD64)]
System  : Windows 11


In [3]:
!pip --version

pip 25.1.1 from C:\Python313\Lib\site-packages\pip (python 3.13)



### Exercise 2 — Install Ollama
Download the installer for your OS from <https://ollama.com/download> (Linux: `curl -fsSL https://ollama.com/install.sh | sh`). Then verify:

In [4]:
!ollama --version

ollama version is 0.40.11


Make sure the Ollama service is running (the desktop app starts it automatically; otherwise run `ollama serve` in a separate terminal).

In [5]:
import requests

OLLAMA = "http://localhost:11434"

try:
    r = requests.get(OLLAMA, timeout=3)
    print("✅ Ollama server reachable:", r.text)
except requests.exceptions.ConnectionError:
    print("❌ Ollama is not running. Start it with `ollama serve` in a terminal.")

✅ Ollama server reachable: Ollama is running


## Part 3 — Your First Local LLM
### Exercise 3 — Download a model
If the exact name differs on your installation, pick a small instruct/chat model suited to your hardware.

In [6]:
# Models used throughout the lab — edit if needed
MODEL  = "qwen2.5:7b"   # Model 1
MODEL2 = "gemma2:2b"    # Model 2 (used in the comparison parts)

In [7]:
!ollama pull {MODEL}

pulling manifest â ‹ pulling manifest â ™ pulling manifest â ¹ pulling manifest â ¸ pulling manifest â ¼ pulling manifest â ´ pulling manifest â ¦ pulling manifest â § pulling manifest â ‡ pulling manifest â � pulling manifest â ‹ pulling manifest â ™ pulling manifest â ¹ pulling manifest â ¸ pulling manifest â ¼ pulling manifest â ´ pulling manifest â ¦ pulling manifest â § pulling manifest â ‡ pulling manifest â � pulling manifest â ‹ pulling manifest â ™ pulling manifest â ¹ pulling manifest â ¸ pulling manifest â ¼ pulling manifest â ´ pulling manifest 
pulling 2bada8a74506: 100% â–•â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–� 4.7 GB                         
pulling 66b9ea09bd5b: 100% â–•â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–�   68 B                         
pulling eb4402837c78: 100% â–•â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–� 1.5 KB                         
pulling 832dd9e00a68: 100% â–•â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ

In [8]:
!ollama list

NAME          ID              SIZE      MODIFIED       
qwen2.5:7b    845dbda0ea48    4.7 GB    11 seconds ago    
gemma2:2b     8ccf136fdd52    1.6 GB    2 hours ago       


### Exercise 4 — Run the model
In the terminal you would use `ollama run qwen3.5`. Here we do the same thing through the API with a helper function.

In [9]:
import time, json

def generate(prompt, model=None, system=None, fmt=None, images=None, options=None, think=None):
    """Call Ollama /api/generate (non-streaming) and return the full JSON dict
    plus the measured wall-clock time under the key 'wall_s'."""
    payload = {"model": model or MODEL, "prompt": prompt, "stream": False}
    if system:  payload["system"] = system
    if fmt:     payload["format"] = fmt          # "json" or a JSON schema
    if images:  payload["images"] = images       # list of base64 strings
    if options: payload["options"] = options
    if think is not None: payload["think"] = think  # False disables 'thinking' on reasoning models
    t0 = time.perf_counter()
    r = requests.post(f"{OLLAMA}/api/generate", json=payload, timeout=600)
    r.raise_for_status()
    out = r.json()
    out["wall_s"] = time.perf_counter() - t0
    return out

def ask(prompt, **kw):
    """Shortcut: return only the text, and print timing."""
    out = generate(prompt, **kw)
    print(f"[{out['wall_s']:.1f}s]")
    return out["response"]

In [10]:
for p in [
    "Explain what artificial intelligence is in five sentences.",
    "Explain artificial intelligence to a 10-year-old child.",
    "Explain artificial intelligence to a computer science professor.",
]:
    print("="*80, "\nPROMPT:", p, "\n")
    print(ask(p))

PROMPT: Explain what artificial intelligence is in five sentences. 

[7.2s]
Artificial intelligence (AI) refers to the simulation of human intelligence in machines that are programmed to think, learn, and perform tasks normally requiring human intelligence, such as visual perception, speech recognition, decision-making, and language translation. AI systems can process large amounts of data, identify patterns, and make decisions or predictions based on that data. Through machine learning, these systems improve their performance over time as they are exposed to more data. AI technologies include natural language processing, robotics, and expert systems, among others.
PROMPT: Explain artificial intelligence to a 10-year-old child. 

[8.7s]
Hey there! Let's talk about something really cool called artificial intelligence, or AI for short. Imagine you have a super-smart friend who can do all sorts of amazing things—like understand what you say, learn from what you tell them, and even help yo

### Questions
1. Did the model produce different answers? Why?
2. Is the model communicating with a cloud API?
3. Where is inference being performed?
4. What happens if the Internet connection is disabled?

> 💡 Try it: disable Wi-Fi / unplug Ethernet and re-run the cell above.

**✍️ Your answer:**

**1. Yes, the model produced completely different answers.**

Why? Because the prompt explicitly defines the target audience and the constraints, which completely changes the model's context:

For the 5 sentences prompt: The model gives a concise, general overview. It focuses on the core definition (simulation of human intelligence) and lists the main subfields (Machine Learning, NLP) without going into deep detail.

For the 10-year-old child: The model uses simple language, analogies, and concrete examples. It compares AI to a "very smart friend" or "super-smart helper" and talks about playing chess or helping doctors. The tone is friendly and pedagogical.

For the CS professor: The model adopts a highly academic and structured tone. It uses Markdown formatting (with ### titles), technical jargon (NLP, Computer Vision, Expert Systems), historical context (AI Winter, John McCarthy), and discusses advanced challenges (Bias, Explainability, Edge Computing).

Conclusion: LLMs are context-prediction engines. By changing the persona ("10-year-old" vs "professor"), we force the model to retrieve different vocabulary, structures, and levels of depth from its training data.



**2. No, the model is NOT communicating with a cloud API.**

Why? The model is running entirely locally on this computer. The Python code sends an HTTP POST request to http://localhost:11434. The word localhost (or IP 127.0.0.1) means "this exact machine". Therefore, the request goes directly to the Ollama service running in the background on my own computer, which then performs the inference using my local CPU/GPU.

Unlike ChatGPT or Claude, no data is sent to the Internet. My prompts and the model's answers never leave my computer. This ensures 100% privacy and means the model could still work even if I disconnected my Wi-Fi.



**3. Inference is being performed entirely on my local machine (my personal computer).**

Where exactly? When the Python script sends the HTTP request to localhost:11434, the Ollama background service receives it and loads the model's weights from the local hard drive into the local RAM and VRAM. The actual mathematical computations (matrix multiplications) required to generate the tokens are executed by my computer's own CPU and/or GPU.

No external servers, data centers, or cloud infrastructures are involved in this process.


**4. What happens if the Internet connection is disabled?**
Nothing changes. The model continues to work perfectly. I disabled my internet connection, re-ran the exact same cell, and the model generated the three answers successfully, just as before.

Why? Because the entire architecture is local. The Python script communicates with localhost:11434 (my own computer), and the inference is performed by my CPU/GPU. No data needs to travel to the internet.

Comparison between the two runs (Online vs. Offline):
There is no functional difference between the answers generated with the internet and those generated without it. Both sets of answers are coherent, well-structured, and adapt perfectly to the target audience (5 sentences, 10-year-old, professor). The only differences are slight variations in the exact wording and response times. This happens because LLMs are probabilistic (they generate text based on probabilities, not fixed rules), so they rarely produce the exact same words twice, even with the same prompt.



## Part 4 — Experiment with Prompts
### Exercise 5 — Prompt engineering

In [11]:
prompt_a = """You are a university professor.

Explain the concept of a Directed Acyclic Graph (DAG).

Your answer must contain:
1. Definition
2. Example
3. Adjacency matrix
4. Real-world application
5. One Python example"""

print(ask(prompt_a))

[17.0s]
Certainly! As a university professor, I would be happy to explain the concept of a Directed Acyclic Graph (DAG) in a clear and educational manner.

### 1. Definition
A Directed Acyclic Graph (DAG) is a directed graph that has no directed cycles. This means that there is no way to start at any vertex (node) and follow a set of edges that will eventually loop back to the starting vertex. In a DAG, the edges have a direction, indicating a one-way relationship between the nodes.

### 2. Example
Imagine a course schedule in a university. You might have a set of courses where certain courses must be completed before others can be taken. For example, you might need to complete Calculus I before you can take Calculus II. If we represent each course as a node and the prerequisite relationship as a directed edge, we would have a DAG.

A simple example could be:
- Node A: Calculus I
- Node B: Calculus II
- Node C: Linear Algebra

The edges could be:
- A → B (Calculus I is a prerequisite f

In [12]:
prompt_b = """You are a university professor.

Explain DAGs to a first-year computer science student.

Use simple language and one concrete example."""

print(ask(prompt_b))

[12.5s]
Sure! Directed Acyclic Graphs, or DAGs, are a type of graph that we use in computer science. Let's break it down:

### What's a Graph?
Imagine you have a bunch of points (we call them nodes) and lines (we call them edges) connecting some of these points. That's a graph!

### What's Directed?
In a directed graph, these lines (edges) have a specific direction. It’s like one-way streets. You can only travel from one point to another in one direction.

### What's Acyclic?
"Acylic" means there are no loops or cycles. It's like saying you can't start at a point and keep going in circles to get back to where you started.

### A Concrete Example
Let's think about a simple example: a course schedule.

Imagine you're a first-year computer science student, and you're trying to decide which courses to take. Each course is represented as a node, and the directed edges represent prerequisites. For example:

1. **CSCI101: Introduction to Computer Science**
2. **CSCI201: Data Structures**
3. *

### Questions
Compare the two answers. Explain the role of: **role · context · constraints · output format · specificity**.

**✍️ Your answer:**

**Comparison of the two answers:**
The two answers are drastically different because the prompts guide the model in very different ways.

Role: Both prompts assigned the model the role of a "university professor". However, in prompt B, the professor is adapting to a beginner's level.

Context: Prompt A implies a formal, academic setting (writing a lecture or a textbook). Prompt B explicitly sets a teaching context ("explain to a first-year student").

Constraints: Prompt A has strict structural constraints (must contain 5 specific items: definition, example, matrix, application, Python code). Prompt B has stylistic constraints (use simple language, use one concrete example).

Output Format: Because of the constraints in A, the output is highly structured with numbered headings, a mathematical-style adjacency matrix, and a Python code block. The output for B is conversational, using analogies (like "one-way streets") and a simple ASCII diagram (CSCI101 → CSCI201).

Specificity: Prompt A is highly specific regarding technical deliverables. Prompt B is specific regarding the target audience.

Conclusion: The more specific and constrained the prompt, the more technical and structured the output will be. If you focus the prompt on the target audience, the model will use better analogies and simpler language.

## Part 5 — Model Management
### Exercise 6 — List, show, and monitor models

In [13]:
!ollama list

NAME          ID              SIZE      MODIFIED       
qwen2.5:7b    845dbda0ea48    4.7 GB    12 minutes ago    
gemma2:2b     8ccf136fdd52    1.6 GB    2 hours ago       


In [14]:
!ollama show {MODEL}

  Model
    architecture        qwen2          
    parameters          7.6B           
    context length      32768          
    embedding length    3584           
    quantization        Q4_K_M         
    template source     go_template    

  Capabilities
    completion    
    tools         

  System
    You are Qwen, created by Alibaba Cloud. You are a helpful assistant.    

  License
    Apache License               
    Version 2.0, January 2004    
    ...                          



In [15]:
!ollama ps

NAME    ID    SIZE    PROCESSOR    CONTEXT    RUNNER    UNTIL 


The same information is available from the REST API (`/api/tags`, `/api/show`, `/api/ps`):

In [17]:
import pandas as pd

def downloaded_models():
    tags = requests.get(f"{OLLAMA}/api/tags").json()["models"]
    return pd.DataFrame([{
        "name": m["name"],
        "size_GB": round(m["size"]/1e9, 2),
        "family": m["details"].get("family"),
        "params": m["details"].get("parameter_size"),
        "quantization": m["details"].get("quantization_level"),
    } for m in tags])

def loaded_models():
    ps = requests.get(f"{OLLAMA}/api/ps").json()["models"]
    return pd.DataFrame([{
        "name": m["name"],
        "size_GB": round(m["size"]/1e9, 2),
        "in_VRAM_GB": round(m.get("size_vram", 0)/1e9, 2),
        "expires_at": m.get("expires_at"),
    } for m in ps])

print("Downloaded:"); display(downloaded_models())
print("Currently loaded in memory:"); display(loaded_models())

Downloaded:


,name,size_GB,family,params,quantization
0,qwen2.5:7b,4.68,qwen2,7.6B,Q4_K_M
1,gemma2:2b,1.63,gemma2,2.6B,Q4_0


Currently loaded in memory:


""


### Questions
- What information do you get from `ollama list` vs `ollama ps`?
- What is the difference between a model that is **downloaded** and one that is **loaded**?

**✍️ Your answer:**

**- What information do you get from ollama list vs ollama ps?**
ollama list shows all the models that are permanently downloaded and stored on the local hard drive (like a library). ollama ps shows only the models that are currently loaded into the system's RAM/VRAM and actively running or ready to answer (like books currently open on a desk).

**What is the difference between a model that is downloaded and one that is loaded?**
A downloaded model sits on the hard drive, taking up disk space (e.g., 4.7 GB) but using 0% of your RAM. A loaded model has been transferred from the disk into the RAM/VRAM by Ollama to perform fast calculations. Loading takes a few seconds, but once loaded, inference is very fast. If you don't use the model for 5 minutes, Ollama unloads it from RAM to free up memory for other applications.


- 


## Part 6 — Compare Two Local Models
Install a second model suited to your computer.

In [18]:
!ollama pull {MODEL2}
!ollama list

pulling manifest â ‹ pulling manifest â ™ pulling manifest â ¹ pulling manifest â ¸ pulling manifest â ¼ pulling manifest â ´ pulling manifest â ¦ pulling manifest â § pulling manifest â ‡ pulling manifest â � pulling manifest â ‹ pulling manifest â ™ pulling manifest â ¹ pulling manifest â ¸ pulling manifest â ¼ pulling manifest â ´ pulling manifest â ¦ pulling manifest 
pulling 7462734796d6: 100% â–•â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–� 1.6 GB                         
pulling e0a42594d802: 100% â–•â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–�  358 B                         
pulling 097a36493f71: 100% â–•â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–� 8.4 KB                         
pulling 2490e7468436: 100% â–•â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–�   65 B                         
pulling e18ad7af7efb: 100% â–•â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–�  487 B                         
verifying sha256 digest 


NAME          ID              SIZE      MODIFIED               
gemma2:2b     8ccf136fdd52    1.6 GB    Less than a second ago    
qwen2.5:7b    845dbda0ea48    4.7 GB    29 minutes ago            


### Exercise 7 — Same prompt, different models

In [19]:
prime_prompt = """Write a Python function that determines whether a number is prime.

Requirements:
- Python
- Type hints
- Docstring
- Explain the algorithm
- Give its time complexity"""

results = {}
for m in (MODEL, MODEL2):
    print("="*80, f"\nMODEL: {m}\n")
    out = generate(prime_prompt, model=m)
    results[m] = out
    print(out["response"])
    print(f"\n⏱ {out['wall_s']:.1f}s")

MODEL: qwen2.5:7b

Certainly! Below is a Python function that determines whether a number is prime. I'll include type hints, a docstring, explain the algorithm, and provide the time complexity.

```python
def is_prime(n: int) -> bool:
    """
    Determine whether a given number n is a prime number.

    A prime number is a natural number greater than 1 that is not a product of two smaller natural numbers.
    This function checks for primality by testing divisibility from 2 up to the square root of n.
    If n is divisible by any number in this range, it is not prime.

    :param n: The number to check for primality.
    :type n: int
    :return: True if n is a prime number, False otherwise.
    :rtype: bool
    """
    # Edge cases
    if n <= 1:
        return False
    if n <= 3:
        return True

    # This is checked so that we can skip middle five numbers in below loop
    if n % 2 == 0 or n % 3 == 0:
        return False

    i = 5
    while i * i <= n:
        if n % i == 0

Now fill in your evaluation (scores out of 5) and see the table:

In [20]:
scores = {
    MODEL:  {"Response quality /5": 5, "Code correctness /5": 5},
    MODEL2: {"Response quality /5": 4, "Code correctness /5": 5},
}  # <- replace None with your scores

table = pd.DataFrame([
    {"Model": m, "Response time (s)": round(results[m]["wall_s"], 1), **scores[m]}
    for m in (MODEL, MODEL2)
])
table

,Model,Response time (s),Response quality /5,Code correctness /5
0,qwen2.5:7b,19.0,5,5
1,gemma2:2b,9.2,4,5


**Tip — test the generated code:** copy the function the model wrote into a cell and check it.

In [21]:
def is_prime_reference(n: int) -> bool:
    if n < 2: return False
    if n < 4: return True
    if n % 2 == 0: return False
    i = 3
    while i * i <= n:
        if n % i == 0: return False
        i += 2
    return True

# Paste the model's function below as `candidate` and compare:
# candidate = is_prime
# assert all(candidate(n) == is_prime_reference(n) for n in range(-5, 2000)), "Incorrect!"
# print("✅ candidate matches the reference on 0..2000")

### Questions
Which model is faster? Which produces better code? Which uses more memory? Which would you choose for programming assistance? Is the largest model necessarily the best?

**✍️ Your answer:**

**Which model is faster?**
The smaller model, gemma2:2b, is much faster (9.2s) compared to qwen2.5:7b (19.0s). This is expected because Gemma has fewer parameters (2B vs 7B), so it requires less computation.

**Which produces better code?**
qwen2.5:7b produced slightly better and more optimized code. It uses the "6k ± 1" optimization trick (incrementing i by 6) to skip unnecessary checks, making the algorithm slightly faster in practice. However, both functions are mathematically correct and return the right results.

**Which uses more memory?**
qwen2.5:7b uses much more memory. It takes 4.7 GB on disk and needs significant RAM/VRAM to run, whereas gemma2:2b only requires 1.6 GB.

**Which would you choose for programming assistance?**
For programming, I would choose qwen2.5:7b. In my experience, larger models (7B+) are generally better at understanding complex instructions, writing optimized code, and explaining algorithms clearly. For quick, simple scripts, however, gemma2:2b is perfect because it is incredibly fast.

**Is the largest model necessarily the best?**
No. The "best" model depends on the hardware and the use case. A larger model offers higher quality but requires more RAM and is slower. A smaller model is faster, lightweight, and can run on almost any laptop, even if it's slightly less "smart". The trade-off is always speed/memory vs. quality.

## Part 7 — Using Local AI from Python
### Exercise 8 — Python + Ollama
Project structure:
```
local_ai_lab/
├── app.py
├── requirements.txt
└── README.md
```
Install `requests` (already available here):

In [22]:
!pip install -q requests


[notice] A new release of pip is available: 25.1.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [24]:
import os
os.makedirs("local_ai_lab", exist_ok=True)

In [25]:
%%writefile local_ai_lab/app.py
import requests

url = "http://localhost:11434/api/generate"

data = {
    "model": "qwen2.5:7b",
    "prompt": "Explain recursion in Python.",
    "stream": False
}

response = requests.post(url, json=data)
result = response.json()
print(result["response"])

Writing local_ai_lab/app.py


In [26]:
%%writefile local_ai_lab/requirements.txt
requests

Writing local_ai_lab/requirements.txt


In [27]:
%%writefile local_ai_lab/README.md
# local_ai_lab
Minimal Python client for a local Ollama server.
Run: `python app.py`

Writing local_ai_lab/README.md


In [28]:
!python local_ai_lab/app.py

Recursion in Python is a method where the solution to a problem depends on solutions to smaller instances of the same problem. In programming, a recursive function is a function that calls itself during its execution. The key aspects of recursion include:

1. **Base Case**: This is the condition that stops the recursion. Without a base case, a recursive function would continue to call itself indefinitely, potentially leading to a stack overflow error.

2. **Recursive Case**: This is the part of the function that breaks the problem into smaller subproblems and calls the function itself.

Here's a simple example of a recursive function in Python that calculates the factorial of a number:

```python
def factorial(n):
    # Base case
    if n == 0 or n == 1:
        return 1
    # Recursive case
    else:
        return n * factorial(n - 1)

# Example usage
print(factorial(5))  # Output should be 120
```

### Example: Sum of Numbers

Here's another example, this time calculating the sum of

**Architecture:** `Python → HTTP POST → localhost:11434 → Ollama → Local LLM → response → Python`

Below is the same call shown step by step, including the **metadata** Ollama returns:

In [29]:
out = generate("Explain recursion in Python.")
print(out["response"][:600], "...\n")
print({k: v for k, v in out.items() if k not in ("response", "context", "thinking")})

Certainly! Recursion in Python (and in programming in general) refers to a method where the solution to a problem depends on solutions to smaller instances of the same problem. In other words, a recursive function calls itself one or more times to solve smaller parts of the problem.

Here are the key components of recursion:

1. **Base Case**: This is the condition that stops the recursion. Without a base case, a recursive function would keep calling itself indefinitely, leading to a stack overflow error.

2. **Recursive Case**: This is the part of the function where the function calls itself  ...

{'model': 'qwen2.5:7b', 'created_at': '2026-10-26T19:16:08.9908431Z', 'done': True, 'done_reason': 'stop', 'total_duration': 14137053000, 'load_duration': 32613200, 'prompt_eval_count': 35, 'prompt_eval_cached_count': 34, 'prompt_eval_duration': 367458000, 'eval_count': 667, 'eval_duration': 13689199000, 'wall_s': 16.18150050000986}


## Part 8 — Build a Local AI Chatbot
### Exercise 9 — Interactive chatbot
`input()` works inside Jupyter (a prompt box appears). Type `exit` to stop.

In [30]:
URL = f"{OLLAMA}/api/generate"

def ask_ai(prompt):
    data = {"model": MODEL, "prompt": prompt, "stream": False}
    response = requests.post(URL, json=data)
    response.raise_for_status()
    return response.json()["response"]

def chat_loop_stateless():
    print("Local AI Assistant\nType 'exit' to stop.")
    while True:
        question = input("\nYou: ")
        if question.lower() == "exit":
            break
        print("\nAI:", ask_ai(question))

# chat_loop_stateless()   # <- uncomment to run

In [31]:
# Non-interactive test (works without typing):
for q in ["What is machine learning?", "Explain reinforcement learning."]:
    print("You:", q); print("AI:", ask_ai(q)[:500], "...\n")

You: What is machine learning?
AI: Machine learning is a subset of artificial intelligence (AI) that focuses on developing algorithms and statistical models that enable computer systems to improve their performance on a specific task through experience or data, without being explicitly programmed. The core idea is to allow software applications to learn from data, identify patterns, and make decisions or predictions with minimal human intervention.

Key aspects of machine learning include:

1. **Data**: Machine learning models ar ...

You: Explain reinforcement learning.
AI: Reinforcement learning is a type of machine learning where an agent learns to make decisions by performing actions in an environment to maximize some notion of cumulative reward. The agent learns by trial and error, receiving feedback in the form of rewards or penalties. This feedback guides the agent in adjusting its actions to maximize its cumulative reward over time.

### Key Concepts in Reinforcement Learning



## Part 9 — Add Conversation Memory
The previous app treats each question independently. We now send the **conversation history** with each request.

In [32]:
history = []

def ask_ai_memory(question):
    history.append(f"User: {question}")
    prompt = "\n".join(history) + "\nAssistant:"
    data = {"model": MODEL, "prompt": prompt, "stream": False}
    response = requests.post(URL, json=data)
    response.raise_for_status()
    answer = response.json()["response"]
    history.append(f"Assistant: {answer}")
    return answer

def chat_loop_memory():
    while True:
        question = input("\nYou: ")
        if question.lower() == "exit":
            break
        print("\nAI:", ask_ai_memory(question))

# chat_loop_memory()   # <- uncomment for interactive use

In [33]:
history.clear()
for q in ["My name is Ahmed.", "What is my name?"]:
    print("You:", q)
    print("AI :", ask_ai_memory(q), "\n")

You: My name is Ahmed.
AI : Hello Ahmed! Nice to meet you. How can I assist you today? 

You: What is my name?
AI : Your name is Ahmed. 



In [34]:
print("What the model actually received on the last call:\n" + "-"*50)
print("\n".join(history[:-1]) + "\nAssistant:")

What the model actually received on the last call:
--------------------------------------------------
User: My name is Ahmed.
Assistant: Hello Ahmed! Nice to meet you. How can I assist you today?
User: What is my name?
Assistant:


### Question
Why does the model remember the name? Is the LLM itself remembering?

**✍️ Your answer:**

**Why does the model remember the name?**

The model "remembers" the name because the Python application is simulating memory. As shown in the printed prompt above, the code takes the entire history list (containing the user's previous message and the assistant's previous reply) and sends it back to the API alongside the new question.

**Is the LLM itself remembering?**

No, absolutely not. The LLM is stateless. It has no internal memory of past requests. Between two calls to the API, the model is completely reset. The illusion of memory is entirely created by the Python script acting as a notepad, writing down the conversation and passing it back to the model every single time.

---
#### Better approach: the `/api/chat` endpoint
Ollama also has a chat endpoint taking structured `messages` (with `system` / `user` / `assistant` roles) and applying the model's own chat template.

In [35]:
def chat(messages, model=None, **kw):
    payload = {"model": model or MODEL, "messages": messages, "stream": False, **kw}
    r = requests.post(f"{OLLAMA}/api/chat", json=payload, timeout=600)
    r.raise_for_status()
    return r.json()

msgs = [{"role": "system", "content": "You are a concise assistant."},
        {"role": "user", "content": "My name is Ahmed."}]
reply = chat(msgs)["message"]
msgs.append(reply)
msgs.append({"role": "user", "content": "What is my name?"})
print(chat(msgs)["message"]["content"])

Your name is Ahmed.


## Part 10 — Local AI for Software Engineering
### Exercise 10 — Code assistant

In [36]:
java_review = """You are a senior Java developer.

Review the following Java code:

public class Calculator {

    public int divide(int a, int b) {
        return a / b;
    }
}

Identify:
1. Potential problems
2. Runtime errors
3. Improvements
4. Corrected code"""

print(ask(java_review))

[11.3s]
Sure, let's review the provided Java code:

```java
public class Calculator {

    public int divide(int a, int b) {
        return a / b;
    }
}
```

### 1. Potential Problems
- **Lack of Error Handling:** The method `divide` does not handle the case where `b` is `0`, which would cause a `ArithmeticException`.
- **Overflow:** Although rare in practice with primitive `int` types, division can result in overflow if `a` is a very large number and `b` is 1. However, this is not a common issue and can be handled, but it is not a critical problem in the context of a simple `int` division.

### 2. Runtime Errors
- **ArithmeticException:** When `b` is 0, the method will throw an `ArithmeticException`.

### 3. Improvements
- **Add Error Handling:** To handle the case when `b` is 0, you should include a check for this condition.
- **Return Type:** Consider returning a `double` for the result to handle fractional results.
- **Method Naming:** The method name `divide` could be more descr

In [37]:
buggy = """You are a senior Java developer.

Find the bug in this code and explain how to fix it:

public class Student {

    private String name;
    private int age;

    public Student(String name) {
        this.name = name;
    }

    public void setAge(int age) {
        age = age;
    }

    public String getName() {
        return name;
    }
}"""

print(ask(buggy))

[8.8s]
The code you provided has a bug in the `setAge` method. The issue is that the `age` parameter is not being assigned to the `age` field of the `Student` class. To fix this, you need to assign the `age` parameter to the `age` field.

Here is the corrected code:

```java
public class Student {

    private String name;
    private int age;

    public Student(String name) {
        this.name = name;
    }

    public void setAge(int age) {
        this.age = age; // Assign the age parameter to the age field
    }

    public String getName() {
        return name;
    }
}
```

### Explanation:

- **Original Code:**
  ```java
  public void setAge(int age) {
      age = age;
  }
  ```
  - In this code, the `age` parameter is being assigned to itself, which does nothing and is redundant. The field `age` is not being updated.

- **Fixed Code:**
  ```java
  public void setAge(int age) {
      this.age = age; // Correct assignment
  }
  ```
  - Here, `this.age` is assigned the value of t

> ✅ Expected finding: `age = age;` assigns the parameter to itself (variable shadowing); it should be `this.age = age;`. Did your model find it? Is `age` also never initialized in the constructor?

## Part 11 — Structured Output
### Exercise 11
First, the naive approach: ask for "ONLY JSON" in the prompt.

In [38]:
json_prompt = """Analyze the following student:

Name: Ahmed
Age: 21
Grade: 15.5
Absences: 2

Return ONLY JSON with this structure:

{
  "name": "...",
  "age": 0,
  "grade": 0,
  "risk": "...",
  "recommendation": "..."
}"""

raw = ask(json_prompt, think=False)
print(raw)

[6.1s]
```json
{
  "name": "Ahmed",
  "age": 21,
  "grade": 15.5,
  "risk": "High",
  "recommendation": "Monitor attendance and provide academic support."
}
```


In [39]:
import re

def extract_json(text):
    """Robust parsing: strip ```json fences and grab the first {...} block."""
    text = re.sub(r"```(?:json)?", "", text).strip()
    m = re.search(r"\{.*\}", text, re.DOTALL)
    return json.loads(m.group(0))

try:
    print(extract_json(raw))
except Exception as e:
    print("❌ Could not parse:", e)

{'name': 'Ahmed', 'age': 21, 'grade': 15.5, 'risk': 'High', 'recommendation': 'Monitor attendance and provide academic support.'}


Better: use Ollama's **structured outputs** — pass a JSON Schema in `format` so the output is constrained to be valid.

In [40]:
schema = {
    "type": "object",
    "properties": {
        "name": {"type": "string"},
        "age": {"type": "integer"},
        "grade": {"type": "number"},
        "risk": {"type": "string", "enum": ["low", "medium", "high"]},
        "recommendation": {"type": "string"},
    },
    "required": ["name", "age", "grade", "risk", "recommendation"],
}

out = generate(json_prompt, fmt=schema, think=False)
student = json.loads(out["response"])
print(json.dumps(student, indent=2))
print("Type of grade:", type(student["grade"]))

{
  "name": "Ahmed",
  "age": 21,
  "grade": 15.5,
  "risk": "medium",
  "recommendation": "Monitor attendance and academic performance closely. Consider providing academic support if necessary."
}
Type of grade: <class 'float'>


### Question
Why is structured output important when integrating an LLM into a software application?

**✍️ Your answer:**

**Why is structured output important when integrating an LLM into a software application?**

Structured output (like JSON) is crucial because software applications cannot easily process free-form text.

Reliability: When an LLM generates free text, it might include conversational filler (e.g., "Here is your JSON:") or forget to close brackets, which breaks the application's code. With a JSON schema, the output is guaranteed to be valid and parsable.

Data Type Enforcement: The schema ensures that numbers are returned as numbers (e.g., grade: 15.5) and not strings, and that specific fields only contain allowed values (e.g., risk: "medium").

Automation: Structured data can be directly fed into a database, an API, or a frontend UI without needing complex, error-prone Regex parsing (like the extract_json function we had to use in the naive approach).

In short, structured output bridges the gap between the unpredictable nature of LLMs and the strict requirements of traditional software.



## Part 12 — Local AI Document Assistant
Create `documents/course.txt`:

In [41]:
os.makedirs("documents", exist_ok=True)
with open("documents/course.txt", "w", encoding="utf-8") as f:
    f.write("""Machine learning is a branch of artificial intelligence.

Supervised learning uses labelled data.

Unsupervised learning works with unlabelled data.

Reinforcement learning learns through interaction
with an environment using rewards and penalties.
""")

### Exercise 12 — Read the document and ask questions

In [42]:
with open("documents/course.txt", "r", encoding="utf-8") as file:
    document = file.read()
print(document)

Machine learning is a branch of artificial intelligence.

Supervised learning uses labelled data.

Unsupervised learning works with unlabelled data.

Reinforcement learning learns through interaction
with an environment using rewards and penalties.



In [43]:
def ask_document(question, document):
    prompt = f"""
Answer the question using only the following document.

DOCUMENT:
{document}

QUESTION:
{question}

If the answer is not contained in the document,
say: "Information not available."
"""
    return ask(prompt, think=False)

print(ask_document("What is reinforcement learning?", document))
print()
print(ask_document("What is quantum computing?", document))

[5.4s]
Reinforcement learning learns through interaction with an environment using rewards and penalties.

[2.2s]
Information not available.


**Observe:** the second answer should be *"Information not available."* — the model is grounded in the supplied document instead of its own training data. Does your model respect the instruction every time?

**✍️ Your observation:**

Yes, the model perfectly respected the instruction. The second answer is exactly "Information not available."

This demonstrates grounding: the model relied exclusively on the provided context rather than its own vast pre-trained knowledge (it definitely knows what quantum computing is). This is crucial in professional applications because it prevents hallucinations (the model inventing false information).



## Part 13 — Mini-RAG
We create four documents, then build a **local retriever** that picks the most relevant one for a question and feeds it to the LLM.

In [44]:
docs = {
"ai.txt": """Artificial intelligence (AI) is the field of computer science that builds systems able to perform tasks normally requiring human intelligence, such as reasoning, perception, language understanding and decision making. AI includes machine learning, expert systems, robotics and natural language processing.""",

"machine_learning.txt": """Machine learning is a branch of artificial intelligence. Supervised learning uses labelled data to learn a mapping from inputs to outputs (classification, regression). Unsupervised learning works with unlabelled data to discover structure such as clusters. Overfitting happens when a model memorises training data and generalises poorly.""",

"reinforcement_learning.txt": """Reinforcement learning learns through interaction with an environment using rewards and penalties. An agent observes a state, chooses an action, and receives a reward. The goal is to learn a policy that maximises the cumulative reward. Q-learning and policy gradients are common algorithms.""",

"deep_learning.txt": """Deep learning uses artificial neural networks with many layers to learn representations from large amounts of data. Convolutional neural networks are used for images, recurrent networks and transformers for sequences and language. Training deep networks usually requires GPUs.""",
}

for name, text in docs.items():
    with open(f"documents/{name}", "w", encoding="utf-8") as f:
        f.write(text)
os.listdir("documents")

['ai.txt',
 'course.txt',
 'deep_learning.txt',
 'machine_learning.txt',
 'reinforcement_learning.txt']

In [45]:
%%writefile rag.py
"""Mini-RAG: a tiny local retriever + prompt builder (keyword TF-IDF, no extra libraries)."""
import math, os, re
from collections import Counter

STOPWORDS = set("""a an the of to in on and or is are was were be by for with as at from that this
it its what which who how why when where do does did can could should would about into using use
learn learns through between such""".split())


def tokenize(text):
    return [w for w in re.findall(r"[a-zA-Z]+", text.lower()) if w not in STOPWORDS]


def load_documents(folder="documents"):
    docs = {}
    for fn in sorted(os.listdir(folder)):
        if fn.endswith(".txt"):
            with open(os.path.join(folder, fn), encoding="utf-8") as f:
                docs[fn] = f.read()
    return docs


class Retriever:
    def __init__(self, docs):
        self.docs = docs
        self.tokens = {n: Counter(tokenize(t)) for n, t in docs.items()}
        df = Counter(w for c in self.tokens.values() for w in c)
        N = len(docs)
        self.idf = {w: math.log((N + 1) / (d + 0.5)) for w, d in df.items()}

    def score(self, question):
        q = tokenize(question)
        return {
            name: sum(self.idf.get(w, 0) * (1 + math.log(c[w])) for w in set(q) if c[w] > 0)
            for name, c in self.tokens.items()
        }

    def top(self, question, k=1, min_score=0.01):
        ranked = sorted(self.score(question).items(), key=lambda x: -x[1])
        return [(n, s) for n, s in ranked[:k] if s >= min_score]

    def context(self, question, k=1):
        hits = self.top(question, k)
        return hits, "\n\n".join(f"[{n}]\n{self.docs[n]}" for n, _ in hits)


def build_prompt(question, context):
    return f"""Answer the question using only the CONTEXT below.
If the answer is not contained in the context, say: "Information not available."

CONTEXT:
{context if context else '(no relevant document found)'}

QUESTION:
{question}
"""

Writing rag.py


In [46]:
import importlib, rag
importlib.reload(rag)

retriever = rag.Retriever(rag.load_documents("documents"))

def rag_answer(question, k=1, model=None):
    hits, ctx = retriever.context(question, k)
    out = generate(rag.build_prompt(question, ctx), model=model, think=False)
    return hits, out["response"], out["wall_s"]

for q in ["What is reinforcement learning?",
          "Which hardware is needed to train neural networks with many layers?",
          "What is supervised learning?",
          "Who won the 2018 World Cup?"]:
    hits, ans, t = rag_answer(q)
    print(f"Q: {q}\n  retrieved: {hits}\n  A: {ans.strip()}\n  ({t:.1f}s)\n")

Q: What is reinforcement learning?
  retrieved: [('course.txt', 1.0831034956175303)]
  A: Reinforcement learning learns through interaction with an environment using rewards and penalties.
  (5.3s)

Q: Which hardware is needed to train neural networks with many layers?
  retrieved: [('deep_learning.txt', 8.42789552798877)]
  A: GPUs
  (2.2s)

Q: What is supervised learning?
  retrieved: [('course.txt', 1.0831034956175303)]
  A: Supervised learning uses labelled data.
  (2.3s)

Q: Who won the 2018 World Cup?
  retrieved: []
  A: Information not available.
  (2.3s)



**Architecture**
```
Documents → Local Retriever → Relevant Context → Ollama → Local LLM → Answer
```
This is the fundamental idea behind **Retrieval-Augmented Generation (RAG)**.

#### Optional extension — semantic retrieval with embeddings
Keyword matching fails when the question uses different words than the document. Ollama can also compute embeddings locally (`ollama pull nomic-embed-text`).

In [47]:
EMBED_MODEL = "nomic-embed-text"
# !ollama pull {EMBED_MODEL}      # <- uncomment once

import numpy as np

def embed(texts):
    r = requests.post(f"{OLLAMA}/api/embed", json={"model": EMBED_MODEL, "input": texts})
    r.raise_for_status()
    return np.array(r.json()["embeddings"])

def semantic_top(question, docs, k=1):
    names = list(docs)
    D = embed([docs[n] for n in names]); q = embed([question])[0]
    sims = D @ q / (np.linalg.norm(D, axis=1) * np.linalg.norm(q))
    return sorted(zip(names, sims), key=lambda x: -x[1])[:k]

# print(semantic_top("What hardware accelerates network training?", rag.load_documents("documents"), k=2))

## Part 14 — Local AI Vision *(optional)*
Requires a **multimodal** model, e.g. `ollama pull gemma3` (vision-capable variants), `llava`, `qwen2.5vl`, `llama3.2-vision`.

In [48]:
VISION_MODEL = MODEL2   # change to a vision-capable model you have installed
IMAGE_PATH = "image.jpg"  # put an image next to the notebook

In [49]:
import base64

def analyze_image(path, model):
    with open(path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    prompt = """Analyze this image.

Identify:
1. Main objects
2. Environment
3. Text visible in the image
4. Possible interpretation"""
    return generate(prompt, model=model, images=[b64])

if os.path.exists(IMAGE_PATH):
    out = analyze_image(IMAGE_PATH, VISION_MODEL)
    print(out["response"], f"\n⏱ {out['wall_s']:.1f}s")
    from IPython.display import Image, display
    display(Image(IMAGE_PATH, width=400))
else:
    print(f"⚠️ Put an image named '{IMAGE_PATH}' next to this notebook, then re-run.")

⚠️ Put an image named 'image.jpg' next to this notebook, then re-run.


### Questions
- Difference between an LLM and a multimodal model?
- What additional information does the vision model receive?
- Which applications could use local vision AI? *(document analysis, OCR, industrial inspection, education, medical-image preprocessing, surveillance analysis…)*

**✍️ Your answer:**

*(write here)*

## Part 15 — Build a Local AI Web Application
Structure:
```
local-ai-web/
├── app.py
├── templates/index.html
└── static/
```

In [50]:
!pip install -q flask requests
os.makedirs("local-ai-web/templates", exist_ok=True)
os.makedirs("local-ai-web/static", exist_ok=True)


[notice] A new release of pip is available: 25.1.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [51]:
%%writefile local-ai-web/app.py
from flask import Flask, render_template, request
import requests

app = Flask(__name__)

OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL = "qwen2.5:7b"


@app.route("/", methods=["GET", "POST"])
def index():
    answer = ""

    if request.method == "POST":
        question = request.form["question"]
        data = {"model": MODEL, "prompt": question, "stream": False}
        response = requests.post(OLLAMA_URL, json=data)
        answer = response.json()["response"]

    return render_template("index.html", answer=answer)


if __name__ == "__main__":
    app.run(debug=True)

Writing local-ai-web/app.py


In [52]:
%%writefile local-ai-web/templates/index.html
<!DOCTYPE html>
<html>
<head>
    <title>Local AI Assistant</title>
</head>
<body>

<h1>Local AI Assistant</h1>

<form method="POST">
    <textarea name="question" rows="8" cols="70" placeholder="Ask your local AI..."></textarea>
    <br><br>
    <button type="submit">Ask</button>
</form>

{% if answer %}
<h2>Response</h2>
<pre>{{ answer }}</pre>
{% endif %}

</body>
</html>

Writing local-ai-web/templates/index.html


Run it **from a terminal** (not inside the notebook):
```bash
cd local-ai-web
python app.py
```
then open <http://127.0.0.1:5000>. 🎉 You now have a complete local AI web application.

*(📸 Take a screenshot for your report: `screenshots/06_webapp.png`.)*

## Part 16 — Performance Benchmark
We measure: response time, model size, tokens generated, tokens/second, RAM and GPU consumption.

Ollama returns precise counters: `eval_count` (generated tokens), `eval_duration` (ns), `prompt_eval_count`, `load_duration`.

In [55]:
!pip install -q psutil


[notice] A new release of pip is available: 25.1.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [56]:
%%writefile benchmark.py
"""Benchmark local Ollama models on the same prompt."""
import time, requests, psutil, pandas as pd

OLLAMA = "http://localhost:11434"
PROMPT = "Write a 500-word explanation of reinforcement learning."


def unload(model):
    requests.post(f"{OLLAMA}/api/generate", json={"model": model, "keep_alive": 0})
    time.sleep(2)


def loaded_info(model):
    for m in requests.get(f"{OLLAMA}/api/ps").json()["models"]:
        if m["name"].startswith(model.split(":")[0]):
            return m["size"] / 1e9, m.get("size_vram", 0) / 1e9
    return float("nan"), float("nan")


def bench(model, prompt=PROMPT, runs=1):
    rows = []
    for _ in range(runs):
        unload(model)                       # cold start, so the load time is visible
        t0 = time.perf_counter()
        r = requests.post(f"{OLLAMA}/api/generate",
                          json={"model": model, "prompt": prompt, "stream": False,
                                "think": False}, timeout=900).json()
        wall = time.perf_counter() - t0
        total_gb, vram_gb = loaded_info(model)
        rows.append({
            "Model": model,
            "Disk size (GB)": next((m["size"] / 1e9 for m in requests.get(f"{OLLAMA}/api/tags").json()["models"]
                                    if m["name"].startswith(model.split(":")[0])), float("nan")),
            "Loaded size (GB)": total_gb,
            "VRAM (GB)": vram_gb,
            "Load time (s)": r.get("load_duration", 0) / 1e9,
            "Response time (s)": wall,
            "Tokens": r.get("eval_count"),
            "Tokens/s": r["eval_count"] / (r["eval_duration"] / 1e9),
            "System RAM used (%)": psutil.virtual_memory().percent,
        })
    return pd.DataFrame(rows)


if __name__ == "__main__":
    import sys
    print(pd.concat([bench(m) for m in sys.argv[1:]]).round(2).to_string(index=False))

Writing benchmark.py


In [57]:
import benchmark; importlib.reload(benchmark)

bench_df = pd.concat([benchmark.bench(m) for m in (MODEL, MODEL2)], ignore_index=True)
bench_df["Quality /5"] = None   # <- fill in after reading both answers
bench_df.round(2)

,Model,Disk size (GB),Loaded size (GB),VRAM (GB),Load time (s),Response time (s),Tokens,Tokens/s,System RAM used (%),Quality /5
0,qwen2.5:7b,4.68,4.76,4.76,3.19,21.12,797,50.33,54.5,None
1,gemma2:2b,1.63,1.93,1.93,1.65,10.55,762,111.49,57.0,None


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 3, figsize=(13, 3.5))
bench_df.plot.bar(x="Model", y="Tokens/s", ax=ax[0], legend=False, title="Speed (tokens/s)", rot=0)
bench_df.plot.bar(x="Model", y="Loaded size (GB)", ax=ax[1], legend=False, title="Memory (GB)", rot=0)
bench_df.plot.bar(x="Model", y="Response time (s)", ax=ax[2], legend=False, title="Latency (s)", rot=0)
plt.tight_layout(); plt.show()

### Discussion — the model-size trade-off
```
Model size ──► Quality
           ──► Memory
           ──► Latency
           ──► Hardware requirements
```
Local model selection should be based on **actual measurements on the target machine**, not just public benchmark rankings.

**✍️ Your discussion:**

**Discussion based on my measured numbers:**

My benchmark clearly illustrates the trade-off between model size and performance.

**Speed vs Size:** The smaller model (gemma2:2b) is incredibly fast, generating 111.49 tokens/s, whereas the larger model (qwen2.5:7b) generates 50.33 tokens/s. This means users interacting with Gemma will get answers twice as fast.

**Memory vs Size:** qwen2.5:7b requires 4.76 GB of VRAM, compared to only 1.93 GB for gemma2:2b. On a computer with limited RAM/VRAM (e.g., 8 GB), the larger model might cause the system to swap memory to the disk, drastically slowing down the entire computer.

**Quality:** Even though Gemma is faster, Qwen (7B) generally provides more accurate, nuanced, and syntactically correct answers (as seen in Part 6).

**conclusion :** The "best" model depends entirely on the use case and the hardware. For a fast, lightweight chatbot on a standard laptop, gemma2:2b is perfect. For complex programming assistance where quality matters more than a few seconds of waiting, qwen2.5:7b is the clear winner

## Part 17 — Final Mini Project: **FSTT Local AI Assistant**

**Functional requirements:** runs entirely locally · uses Ollama + a local LLM · web interface · answers questions about a supplied course · uses local documents as knowledge · maintains conversation history · structured responses for selected questions · displays the answer · records response time.

```
Browser → Flask/Python ─┬─ Conversation history ─┐
                        └─ Documents → Retriever ┴─► Ollama → Local LLM
```

We reuse `rag.py` and create the final deliverable layout:
```
LocalAI_Lab/
├── source/ (app.py, chatbot.py, rag.py, benchmark.py, templates/)
├── documents/
├── screenshots/
├── report.pdf
└── README.md
```

In [58]:
import shutil
base = "LocalAI_Lab"
for d in ("source/templates", "documents", "screenshots"):
    os.makedirs(f"{base}/{d}", exist_ok=True)

# copy shared modules and documents into the deliverable layout
shutil.copy("rag.py", f"{base}/source/rag.py")
shutil.copy("benchmark.py", f"{base}/source/benchmark.py")
for fn in os.listdir("documents"):
    shutil.copy(f"documents/{fn}", f"{base}/documents/{fn}")
print(sorted(os.listdir(base)))

['documents', 'screenshots', 'source']


In [60]:
%%writefile LocalAI_Lab/source/chatbot.py
"""Console chatbot with conversation memory (uses /api/chat)."""
import requests

OLLAMA = "http://localhost:11434"
MODEL = "qwen2.5:7b"


def main():
    messages = [{"role": "system", "content": "You are a helpful, concise assistant."}]
    print("Local AI Assistant — type 'exit' to stop.")
    while True:
        q = input("\nYou: ")
        if q.lower() == "exit":
            break
        messages.append({"role": "user", "content": q})
        r = requests.post(f"{OLLAMA}/api/chat",
                          json={"model": MODEL, "messages": messages, "stream": False})
        r.raise_for_status()
        reply = r.json()["message"]
        messages.append({"role": "assistant", "content": reply["content"]})
        print("\nAI:", reply["content"])


if __name__ == "__main__":
    main()

Overwriting LocalAI_Lab/source/chatbot.py


In [61]:
%%writefile LocalAI_Lab/source/app.py
"""FSTT Local AI Assistant — Flask + Ollama + local RAG + conversation history."""
import json, os, time
import requests
from flask import Flask, jsonify, render_template, request

import rag

OLLAMA = os.environ.get("OLLAMA_URL", "http://localhost:11434")
MODEL = os.environ.get("MODEL", "qwen2.5:7b")
DOC_DIR = os.path.join(os.path.dirname(__file__), "..", "documents")
MAX_TURNS = 6                      # remembered user/assistant pairs

app = Flask(__name__)
retriever = rag.Retriever(rag.load_documents(DOC_DIR))
history = []                       # [{"role": ..., "content": ...}]  (single-user demo)

SYSTEM = ("You are the FSTT Local AI Assistant. Answer questions about the course using only the "
          "provided CONTEXT. If the answer is not in the context, say: \"Information not available.\" "
          "Be concise.")

# Structured response schema used for "Analyze ..." / "Quiz ..." style questions
STRUCT_SCHEMA = {
    "type": "object",
    "properties": {
        "topic": {"type": "string"},
        "summary": {"type": "string"},
        "key_points": {"type": "array", "items": {"type": "string"}},
        "difficulty": {"type": "string", "enum": ["easy", "medium", "hard"]},
    },
    "required": ["topic", "summary", "key_points", "difficulty"],
}


def call_ollama(messages, structured=False):
    payload = {"model": MODEL, "messages": messages, "stream": False, "think": False}
    if structured:
        payload["format"] = STRUCT_SCHEMA
    r = requests.post(f"{OLLAMA}/api/chat", json=payload, timeout=600)
    r.raise_for_status()
    return r.json()


@app.route("/")
def index():
    return render_template("index.html", model=MODEL, docs=list(retriever.docs))


@app.route("/ask", methods=["POST"])
def ask():
    body = request.get_json(force=True)
    question = (body.get("question") or "").strip()
    structured = bool(body.get("structured"))
    if not question:
        return jsonify(error="Empty question"), 400

    hits, context = retriever.context(question, k=2)
    user_msg = f"CONTEXT:\n{context or '(none)'}\n\nQUESTION: {question}"
    messages = [{"role": "system", "content": SYSTEM}] + history[-2 * MAX_TURNS:] \
               + [{"role": "user", "content": user_msg}]

    t0 = time.perf_counter()
    try:
        out = call_ollama(messages, structured)
    except requests.RequestException as e:
        return jsonify(error=f"Ollama error: {e}"), 502
    elapsed = time.perf_counter() - t0

    answer = out["message"]["content"]
    # store the plain question (not the long context) to keep history small
    history.append({"role": "user", "content": question})
    history.append({"role": "assistant", "content": answer})

    parsed = None
    if structured:
        try:
            parsed = json.loads(answer)
        except json.JSONDecodeError:
            pass

    tokens = out.get("eval_count")
    tps = tokens / (out["eval_duration"] / 1e9) if tokens and out.get("eval_duration") else None
    return jsonify(answer=answer, structured=parsed,
                   sources=[{"doc": n, "score": round(s, 2)} for n, s in hits],
                   seconds=round(elapsed, 2), tokens=tokens,
                   tokens_per_s=round(tps, 1) if tps else None)


@app.route("/reset", methods=["POST"])
def reset():
    history.clear()
    return jsonify(ok=True)


if __name__ == "__main__":
    # Bind to localhost ONLY (see report question Q15 on security)
    app.run(host="127.0.0.1", port=int(os.environ.get("PORT", 5001)), debug=False)

Writing LocalAI_Lab/source/app.py


In [62]:
%%writefile LocalAI_Lab/source/templates/index.html
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<title>FSTT Local AI Assistant</title>
<style>
  body { font-family: system-ui, sans-serif; max-width: 820px; margin: 2rem auto; padding: 0 1rem; color:#222; }
  h1 { margin-bottom: .2rem; } .sub { color:#666; font-size:.9rem; margin-bottom:1rem; }
  #chat { border:1px solid #ddd; border-radius:8px; padding:1rem; height:420px; overflow-y:auto; background:#fafafa; }
  .msg { margin:.6rem 0; padding:.6rem .8rem; border-radius:8px; white-space:pre-wrap; }
  .user { background:#e3f0ff; } .ai { background:#fff; border:1px solid #e5e5e5; }
  .meta { font-size:.75rem; color:#777; margin-top:.3rem; }
  form { display:flex; gap:.5rem; margin-top:1rem; } input[type=text] { flex:1; padding:.6rem; }
  button { padding:.6rem 1rem; cursor:pointer; } label { font-size:.85rem; }
</style>
</head>
<body>
<h1>🎓 FSTT Local AI Assistant</h1>
<div class="sub">Model: <b>{{ model }}</b> · Documents: {{ docs|join(", ") }} · Runs 100% locally</div>
<div id="chat"></div>
<form id="f">
  <input type="text" id="q" placeholder="Ask a question about the course..." autocomplete="off" autofocus>
  <button type="submit">Ask</button>
  <button type="button" id="reset">Reset</button>
</form>
<label><input type="checkbox" id="struct"> Structured response (JSON)</label>

<script>
const chat = document.getElementById("chat");
function add(cls, text, meta) {
  const d = document.createElement("div"); d.className = "msg " + cls; d.textContent = text;
  if (meta) { const m = document.createElement("div"); m.className = "meta"; m.textContent = meta; d.appendChild(m); }
  chat.appendChild(d); chat.scrollTop = chat.scrollHeight; return d;
}
document.getElementById("f").onsubmit = async (e) => {
  e.preventDefault();
  const q = document.getElementById("q"); const question = q.value.trim(); if (!question) return;
  q.value = ""; add("user", question);
  const wait = add("ai", "…thinking…");
  try {
    const r = await fetch("/ask", {method:"POST", headers:{"Content-Type":"application/json"},
      body: JSON.stringify({question, structured: document.getElementById("struct").checked})});
    const j = await r.json(); wait.remove();
    if (j.error) { add("ai", "⚠️ " + j.error); return; }
    const text = j.structured ? JSON.stringify(j.structured, null, 2) : j.answer;
    const src = j.sources.map(s => s.doc).join(", ") || "none";
    add("ai", text, `⏱ ${j.seconds}s · ${j.tokens ?? "?"} tokens · ${j.tokens_per_s ?? "?"} tok/s · sources: ${src}`);
  } catch (err) { wait.remove(); add("ai", "⚠️ " + err); }
};
document.getElementById("reset").onclick = async () => { await fetch("/reset", {method:"POST"}); chat.innerHTML = ""; };
</script>
</body>
</html>

Writing LocalAI_Lab/source/templates/index.html


In [63]:
%%writefile LocalAI_Lab/README.md
# FSTT Local AI Assistant

A fully local assistant: Flask web UI → Ollama → local LLM, with a keyword (TF-IDF) retriever over `documents/`,
conversation history, structured (JSON) answers and response-time display.

## Setup
```bash
ollama pull qwen3.5
pip install flask requests psutil pandas
```

## Run
```bash
cd source
python app.py            # http://127.0.0.1:5001
python chatbot.py        # console chatbot
python benchmark.py qwen3.5 gemma3
```
Environment variables: `MODEL`, `OLLAMA_URL`, `PORT`.

Writing LocalAI_Lab/README.md


**Test the final app from the notebook:** launch it as a background process, send a few questions, then stop it.

In [67]:
import subprocess, sys, time

env = {**os.environ, "MODEL": MODEL, "PORT": "5001"}
server = subprocess.Popen([sys.executable, "app.py"], cwd="LocalAI_Lab/source", env=env,
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
time.sleep(3)

def app_ask(q, structured=False):
    r = requests.post("http://127.0.0.1:5001/ask", json={"question": q, "structured": structured}, timeout=600)
    return r.json()

try:
    for q, s in [("What is reinforcement learning?", False),
                 ("And how is it different from supervised learning?", False),   # tests history
                 ("Summarize deep learning", True),                              # structured
                 ("What is quantum computing?", False)]:                         # not in the docs
        j = app_ask(q, s)
        print(f"Q: {q}\nA: {j.get('structured') or j.get('answer') or j}\n"
              f"   sources={j.get('sources')} · {j.get('seconds')}s · {j.get('tokens_per_s')} tok/s\n")
finally:
    pass  # keep running so you can open http://127.0.0.1:5001 — stop in the next cell

Q: What is reinforcement learning?
A: Reinforcement learning learns through interaction with an environment using rewards and penalties. An agent observes a state, chooses an action, and receives a reward. The goal is to learn a policy that maximises the cumulative reward. Common algorithms include Q-learning and policy gradients.
   sources=[{'doc': 'course.txt', 'score': 1.08}, {'doc': 'reinforcement_learning.txt', 'score': 1.02}] · 6.32s · 48.6 tok/s

Q: And how is it different from supervised learning?
A: Reinforcement learning learns through interaction with an environment using rewards and penalties, whereas supervised learning uses labelled data to learn a mapping from inputs to outputs.
   sources=[{'doc': 'course.txt', 'score': 1.08}, {'doc': 'machine_learning.txt', 'score': 1.06}] · 2.78s · 52.8 tok/s

Q: Summarize deep learning
A: {'difficulty': 'medium', 'key_points': ['Artificial neural networks with many layers', 'Learns from large amounts of data', 'Includes convolutiona

In [68]:
server.terminate(); server.wait(timeout=10)
print("Server stopped.")

Server stopped.


> 📸 Open <http://127.0.0.1:5001> in your browser while the server is running and take your screenshots: `01_installation.png`, `02_model.png`, `03_chatbot.png`, `04_python.png`, `05_rag.png`, `06_webapp.png`.

## Part 18 — Questions for the Report
Answer each question in your own words (use your measurements from Parts 6 and 16 for Q13–Q14).

**Q1.** What is Local AI?

**Local AI refers to running artificial intelligence models directly on a user's personal device (computer, laptop, or smartphone) rather than relying on a remote cloud server. The data, computation, and model weights remain entirely on the local hardware.
**

**Q2.** Difference between a Cloud LLM and a Local LLM?

**A Cloud LLM (like ChatGPT) runs on massive data centers. It requires an internet connection, sends your data to a third party, and usually charges per token, but offers state-of-the-art performance. A Local LLM runs on your own hardware (CPU/GPU). It guarantees 100% privacy, works offline, and is free to use, but is limited by your computer's RAM and processing speed.
**
**Q3.** What role does Ollama play?

**Ollama acts as a local server and model manager. It downloads the models, optimizes them for your hardware (handling the GPU/CPU split), and exposes them via a REST API on localhost:11434. This allows any application (Python, Web apps) to easily communicate with the local LLM.
**

**Q4.** What is model quantization?

**Quantization is a compression technique that reduces the size of the model. Instead of storing every weight (parameter) with 16-bit or 32-bit precision, it stores them using 4-bit or 8-bit precision (e.g., Q4_K_M seen in ollama show). This drastically reduces the RAM required to run the model, with a very small loss in accuracy.
**

**Q5.** Why does model size affect memory consumption?

**A model's size (e.g., 7B parameters) determines how much RAM/VRAM is needed. Each parameter must be loaded into memory. A 7B model at 4-bit quantization requires about 4.7 GB of RAM just to load the weights, whereas a 2B model requires only 1.6 GB. Larger models need more memory to perform the mathematical operations.
**

**Q6.** Why can a smaller model sometimes be preferable?

**As demonstrated in my benchmark (Part 16), a smaller model like gemma2:2b is dramatically faster (111 tokens/s) and uses less than half the memory of a 7B model. A smaller model is preferable when:

The hardware is limited (e.g., a standard laptop with 8 GB of RAM).

Low latency is critical (e.g., real-time chatbots, autocomplete).

The task is simple (e.g., basic text classification or short summarization), where the advanced reasoning of a larger model isn't necessary.
**
**Q7.** Advantages of local inference?

**Absolute Privacy: No data is sent to a third-party cloud. This is crucial for handling sensitive information (medical, legal, corporate data).

Offline Operation: Works perfectly without an internet connection.

Zero Marginal Cost: No API fees or subscription limits. You can generate millions of tokens for free (only paying for electricity).

Full Control: You own the model and the data pipeline.
**

**Q8.** Disadvantages of local inference?

Hardware Constraints: Requires significant RAM/VRAM and a decent CPU/GPU. You cannot run massive 70B+ models on a standard computer.

Lower Capability: Even the best 7B model is generally less "smart" than cloud giants like GPT-4 or Claude 3.5.

Maintenance: The user is responsible for downloading models, managing storage space, and updating the Ollama server.

Speed: Can be slower than cloud data centers if you don't have a dedicated high-end GPU.


**Q9.** Explain the architecture of your Python application.

The architecture follows a local client-server model (built in Part 17):

The Frontend (HTML/JS in the browser) sends a question to the Flask server.

The Flask Backend receives the question and uses a TF-IDF Retriever to search local .txt files and extract the most relevant context.

It constructs a prompt (combining context, conversation history, and the question) and sends an HTTP POST request to the Ollama API (localhost:11434).

Ollama runs the local LLM to generate the response.

The response (along with performance metrics) is sent back to Flask and displayed in the web interface.

**Q10.** Explain the role of the REST API.

The REST API (exposed by Ollama on localhost:11434) acts as a standardized communication bridge. It allows our Python code to interact with the complex C++ inference engine of Ollama using simple HTTP requests. By sending a JSON payload with parameters (model, prompt, stream) to endpoints like /api/generate or /api/chat, the application receives a structured JSON response with the generated text and performance metrics (eval_count, eval_duration). This decouples the UI from the AI engine.

**Q11.** What is RAG?

RAG stands for Retrieval-Augmented Generation. It is an architectural pattern that enhances the accuracy and reliability of LLMs. Instead of relying solely on the model's internal training data (which can lead to hallucinations), a RAG system first retrieves relevant documents from a local knowledge base (using a retriever like TF-IDF or vector search). It then injects this retrieved context into the prompt sent to the LLM. The LLM is instructed to answer using only this context. As demonstrated in Part 13, this ensures the AI answers based on specific, up-to-date, or private documents without needing to be retrained.


**Q12.** Difference between an LLM and a multimodal LLM?

A standard LLM (Large Language Model) is trained exclusively on text. It takes text as input and generates text as output. A multimodal LLM (like LLaVA or GPT-4V) is trained on multiple data types (modalities) simultaneously—typically text and images. It uses an image encoder (like a Vision Transformer) to process pixels into embeddings that the language model can understand. This allows the model to describe images, read handwritten text, or answer questions based on visual input.

**Q13.** Compare two local models experimentally.

(Based on the benchmark performed in Part 16)
I compared qwen2.5:7b and gemma2:2b:

Speed: gemma2:2b was significantly faster, generating 111.49 tokens/s compared to 50.33 tokens/s for qwen2.5:7b.

Memory: qwen2.5:7b required 4.76 GB of VRAM, whereas gemma2:2b used only 1.93 GB.

Latency: gemma2:2b responded in 10.55s, while qwen2.5:7b took 21.12s.

Quality: In Part 6 (Prime number code), qwen2.5:7b provided a more optimized algorithm, proving that larger models generally have better reasoning capabilities. However, gemma2:2b still provided a perfectly functional and correct answer.


**Q14.** Which model would you deploy on a computer with limited RAM? Explain.

I would deploy gemma2:2b (or an even smaller model like qwen2.5:1.5b).
Explanation: On a computer with limited RAM (e.g., 8 GB total), deploying qwen2.5:7b would consume nearly 5 GB just for the model weights, leaving very little memory for the operating system, the web browser, and the Python/Flask application. This would cause "swapping" (using the hard drive as RAM), which drastically slows down the entire system. gemma2:2b uses only 1.9 GB, ensuring the system remains responsive while still providing fast and acceptable AI capabilities.

**Q15.** What security risks exist when exposing a local LLM server to a network?

Several major risks exist:

Unauthorized Compute Use: By default, Ollama has no authentication. If exposed to a network (OLLAMA_HOST=0.0.0.0), anyone on the network (or the internet if the port is forwarded) can use your GPU to mine data or generate spam.

Data Exposure: Users could query the LLM and potentially extract sensitive data from the context window or the server's system prompts.

Prompt Injection: Malicious users could craft prompts to bypass system instructions (e.g., to make the RAG system ignore its safety rules or output harmful content).

Denial of Service (DoS): Sending massive requests could exhaust your RAM/VRAM, crashing your computer.

Mitigation: A secure deployment requires placing Ollama behind a reverse proxy (like Nginx) configured with Authentication (API keys/Passwords) and TLS/HTTPS encryption, or restricting access strictly via a Firewall and VPN.

*(Hint: Ollama has no authentication by default; think about `OLLAMA_HOST=0.0.0.0`, unauthorized use of your compute, model/data exposure, prompt injection through documents, and what a reverse proxy with auth/TLS or a firewall would change.)*

## Part 19 — Deliverables
Final check of the submission folder:
```
LocalAI_Lab/
├── report.pdf
├── source/        app.py · chatbot.py · rag.py · benchmark.py
├── documents/     ai.txt · ml.txt · rl.txt  (+ the other course files)
├── screenshots/   01_installation.png … 06_webapp.png
└── README.md
```

In [74]:
required = [
    "LocalAI_Lab/report.pdf",
    "LocalAI_Lab/README.md",
    "LocalAI_Lab/source/app.py", "LocalAI_Lab/source/chatbot.py",
    "LocalAI_Lab/source/rag.py", "LocalAI_Lab/source/benchmark.py",
    "LocalAI_Lab/screenshots/01_installation.png", "LocalAI_Lab/screenshots/02_model.png",
    "LocalAI_Lab/screenshots/03_chatbot.png", "LocalAI_Lab/screenshots/04_python.png",
    "LocalAI_Lab/screenshots/05_rag.png", "LocalAI_Lab/screenshots/06_webapp.png",
]
for p in required:
    print("✅" if os.path.exists(p) else "⬜ missing", p)
print("\nDocuments:", sorted(os.listdir("LocalAI_Lab/documents")))

✅ LocalAI_Lab/report.pdf
✅ LocalAI_Lab/README.md
✅ LocalAI_Lab/source/app.py
✅ LocalAI_Lab/source/chatbot.py
✅ LocalAI_Lab/source/rag.py
✅ LocalAI_Lab/source/benchmark.py
✅ LocalAI_Lab/screenshots/01_installation.png
✅ LocalAI_Lab/screenshots/02_model.png
✅ LocalAI_Lab/screenshots/03_chatbot.png
✅ LocalAI_Lab/screenshots/04_python.png
✅ LocalAI_Lab/screenshots/05_rag.png
✅ LocalAI_Lab/screenshots/06_webapp.png

Documents: ['ai.txt', 'course.txt', 'deep_learning.txt', 'machine_learning.txt', 'reinforcement_learning.txt']
